# Hero Analysis Notebook

Verifying the flattened Hero tables using rich visualization.

In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from super.core.runtime import bootstrap_spark_env
from super.core import utils
from super.core.display import display_scrollable_dataframe

# Bootstrap & Session
bootstrap_spark_env()
spark = SparkSession.builder.appName("HeroAnalyzer").getOrCreate()

# Config
conf = utils.get_app_conf("generate_powers")
warehouse_root = os.path.join(conf.get_string("stage_root"), "warehouse")
print(f"Reading Warehouse: {warehouse_root}")

## 1. Hero Genome Reports (Unified View)
Analysis of the **Gene Cluster** for each hero.
The table shows the **Master Regulator** (Root Gene) and the summarization of the entire **Cluster** (Network) it controls.

In [ ]:
# Load Tables
hero_profiles = spark.read.parquet(os.path.join(warehouse_root, "hero_profiles"))
hero_genes = spark.read.parquet(os.path.join(warehouse_root, "hero_genes"))
hero_reg = spark.read.parquet(os.path.join(warehouse_root, "hero_gene_regulation"))

# 1. Aggregate Cluster: Create a summary string of regulated genes
# Format: "GeneID(Effect:Strength)"
reg_summary = hero_reg.withColumn(
    "link_desc", 
    F.format_string("%s(%s:%.2f)", F.col("target_gene_id"), F.col("effect"), F.col("strength"))
).groupBy("hero_name").agg(
    F.count("target_gene_id").alias("cluster_size"),
    F.concat_ws(", ", F.collect_list("link_desc")).alias("cluster_network_summary")
)

# 2. Join Everything
p = hero_profiles.alias("p")
g = hero_genes.alias("g")
r = reg_summary.alias("r")

full_report = p.join(g, "hero_name", "left") \
    .join(r, "hero_name", "left") \
    .select(
        F.col("p.hero_name"),
        F.col("p.ontology"),
        F.col("p.bio"),
        F.col("g.gene_id").alias("master_regulator_id"),
        F.col("g.mutation_class"),
        F.col("r.cluster_size"),
        F.col("r.cluster_network_summary")
    )

# Display as a Form-like table
display_scrollable_dataframe(full_report.orderBy("hero_name").toPandas())

## 2. Deep Dive: Bugs Bunny's Cluster

In [ ]:
bugs_report = full_report.filter(F.col("hero_name") == "Bugs Bunny")
# Show full text for Bugs to verify richness
for row in bugs_report.collect():
    print(f"HERO: {row['hero_name']} ({row['ontology']})")
    print(f"BIO: {row['bio']}")
    print(f"MASTER REGULATOR: {row['master_regulator_id']} ({row['mutation_class']})")
    print(f"CLUSTER SIZE: {row['cluster_size']} regulated genes")
    print(f"CLUSTER MAP: {row['cluster_network_summary']}")
